# M5 Exploration — Chronos-2 & TFT
Runs Chronos-2 (zero-shot and fine-tuned) and TFT via `m5_exploration.py`.
Forecasts are cached as parquet — rerunning a cell reloads from disk unless `force_run=True`.

# 1 · Imports

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA device count: 2
Current device: 0
Device name: Quadro RTX 5000


In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.9.1+cu128
12.8


In [3]:
import os, torch, json, gc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from m5_exploration import (
    M5ExplorationSuite,
    DEFAULT_CHRONOS_CONFIG,
    DEFAULT_TFT_CONFIG,
    ALL_KNOWN_COV_COLS,
    ALL_STATIC_COLS,
)
from m5_evaluator import trim_series_to_active, M5Evaluator


print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


/home/nmwamsojo/tsfm-explo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU: Quadro RTX 5000


In [4]:
# Access the operating system interface to modify environment variables
import os

#os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# Import the PyTorch library to manage model tensors and execution
import torch

# Create a torch.device object explicitly targeting the CPU for all model operations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# 2 · Config
**Edit this cell only** to change experiments. Everything below reads from these dicts.

In [12]:
# ── Data ──────────────────────────────────────────────────────────────────────
DATA_PATH    = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"
CUTOFF_DAY   = "2016-05-22"   # evaluation phase cutoff (d_1941)
DATA_TAG     = "default" # "sales_events"       # must match the M5DataPipeline tag used to cache hist_df

# get context length
clen = 512

CUTOFF_DAY = pd.to_datetime(CUTOFF_DAY) - pd.Timedelta(days=28)
CUTOFF_DAY = CUTOFF_DAY.strftime("%Y-%m-%d")    
print(f"Cutoff day: {CUTOFF_DAY}")

# ── Experiment tags ────────────────────────────────────────────────────────────
# Each tag becomes a subfolder: <base_dir>/<data_tag>/level_12/<cutoff>/models/<tag>/
# Change tag when you change config so results don't overwrite each other.
EXP_TAGS = {
    "chronos_zeroshot":      f"chronos2_base_zeroshot_cl{clen}",
    #"chronos_zeroshot_long": "chronos2_base_zeroshot_ctx1913",
    #"chronos_finetune":      "chronos2_base_ft500_lora",
    #"chronos_cov":           "chronos2_base_ft500_cov_snap_events",
    #"tft_vanilla":           "tft_ctx56_h64",
    #"tft_static":            "tft_ctx56_h128_static",
    #"tft_full_cov":          "tft_ctx84_h128_static_allcov",
}

# ── AutoGluon wrapper settings (shared by all experiments) ────────────────────
WRAPPER = {
    "eval_metric":         "RMSSE",
    "enable_ensemble":     False,
    "skip_model_selection": True,
    "verbosity":           1,
}

# ── Per-experiment model configs ───────────────────────────────────────────────
# Start from the defaults imported above and override only what changes.
# Keys not listed here fall back to the defaults.

# --- Chronos zero-shot (no training, fastest) ---------------------------------
CFG_CHRONOS_ZEROSHOT = {
    **DEFAULT_CHRONOS_CONFIG,
    "context_length":  clen,
    "fine_tune_steps": 0,        # zero-shot: no gradient updates
    "known_cov_cols":  [],        # zeroshot ignores covariates
    "use_static":      False,
}

# --- Chronos fine-tuned (LoRA, no covariates) ---------------------------------
CFG_CHRONOS_FINETUNE = {
    **DEFAULT_CHRONOS_CONFIG,
    "context_length":  512,
    "fine_tune_steps": 500,       # try 100, 200, 500, 1000
    "fine_tune_mode":  "lora",    # "lora" (fast) or "full" (best, needs big GPU)
    "fine_tune_lr":    1e-4,
    "known_cov_cols":  [],
    "use_static":      False,
}


# --- TFT vanilla (no static, no covariates) -----------------------------------
CFG_TFT_VANILLA = {
    **DEFAULT_TFT_CONFIG,
    "context_length": 56,
    "hidden_size":    64,
    "known_cov_cols": [],         # no covariates
    "use_static":     False,
}


# Map tag → config for batch running (Section 5)
ALL_EXPERIMENTS = {
    EXP_TAGS["chronos_zeroshot"]:      ("Chronos2", CFG_CHRONOS_ZEROSHOT),
    #EXP_TAGS["chronos_zeroshot_long"]: ("Chronos2", CFG_CHRONOS_ZEROSHOT_LONG),
    #EXP_TAGS["chronos_finetune"]:      ("Chronos2", CFG_CHRONOS_FINETUNE),
    #EXP_TAGS["chronos_cov"]:           ("Chronos2", CFG_CHRONOS_COV),
    #EXP_TAGS["tft_vanilla"]:           ("TFT",      CFG_TFT_VANILLA),
    #EXP_TAGS["tft_static"]:            ("TFT",      CFG_TFT_STATIC),
    #EXP_TAGS["tft_full_cov"]:          ("TFT",      CFG_TFT_FULL),
}

print("Config loaded. Experiments defined:", list(EXP_TAGS.values()))

Cutoff day: 2016-04-24
Config loaded. Experiments defined: ['chronos2_base_zeroshot_cl512']


In [13]:
DEFAULT_CHRONOS_CONFIG

{'model_path': 'autogluon/chronos-2',
 'context_length': 128,
 'fine_tune_steps': 0,
 'fine_tune_mode': 'lora',
 'fine_tune_batch_size': 256,
 'device': 'cuda',
 'batch_size': 256,
 'num_samples': 20,
 'known_cov_cols': [],
 'past_cov_cols': [],
 'use_static': False}

# 3 · Data loading

In [6]:
# ── Reuse the same M5DataPipeline class from the benchmarks notebook ──────────
import os, numpy as np, pyarrow.parquet as pq

class M5DataPipeline:
    """Unchanged from m5_run_benchmarks.ipynb — do not edit here."""
    def __init__(self, config, target_col="sales_quantity"):
        self.target = target_col
        self.date   = "date"
        self.id     = "id"
        self.base_dir   = "/mnt/lab/nmwamsojo/prepared_data"
        self.tag        = config.get("tag", "default")
        self.model_tag  = config.get("model_tag", "default_model")
        self.level_map  = {
            1: [], 2: ["state_id"], 3: ["store_id"], 4: ["cat_id"], 5: ["dept_id"],
            6: ["state_id", "cat_id"], 7: ["state_id", "dept_id"],
            8: ["store_id", "cat_id"], 9: ["store_id", "dept_id"],
            10: ["item_id"], 11: ["state_id", "item_id"], 12: ["id"]
        }
        self.covariates  = ["wm_yr_wk", "wday", "month", "year",
                            "event_name_1", "event_type_1",
                            "snap_CA", "snap_TX", "snap_WI", "sell_price"]
        self.static_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

    def _get_base_folder(self, cutoff_day, level):
        return os.path.join(self.base_dir, self.tag,
                            f"level_{level}", cutoff_day.replace("-", ""))

    def _get_cache_paths(self, cutoff_day, level):
        folder = self._get_base_folder(cutoff_day, level)
        return {"folder": folder,
                "hist":   os.path.join(folder, "hist.parquet"),
                "future": os.path.join(folder, "future.parquet"),
                "static": os.path.join(folder, "static.parquet")}
    
    def optimize_dtypes(self, df):
        # Sales are always small integers in M5
        if "sales_quantity" in df.columns:
            df["sales_quantity"] = pd.to_numeric(df["sales_quantity"], downcast="float")
        # Prices and SNAP flags
        for col in df.select_dtypes(include=['float64']).columns:
            df[col] = pd.to_numeric(df[col], downcast="float")
        for col in df.select_dtypes(include=['int64']).columns:
            df[col] = pd.to_numeric(df[col], downcast="integer")
        
        if "id" in df.columns:
            df["id"] = df["id"].astype("category")
        return df
    
    def get_forecast_paths(self, cutoff_day, level, model_tag=None):
        m_tag  = model_tag or self.model_tag
        folder = os.path.join(self._get_base_folder(cutoff_day, level), "models", m_tag)
        return {"folder":   folder,
                "forecast": os.path.join(folder, "forecasts.parquet"),
                "metrics":  os.path.join(folder, "metrics.json")}
    

    def compute_smoothness_segments(self, hist_df: pd.DataFrame) -> pd.DataFrame:
        # 1. Filter for non-zero demand once to speed up stats
        nz_df = hist_df[hist_df["sales_quantity"] > 0].copy()

        # 2. Group by ID
        g = hist_df.groupby("id")["sales_quantity"]
        gnz = nz_df.groupby("id")["sales_quantity"]

        # 3. Calculate ADI (Average Demand Interval)
        total_periods = g.size()
        non_zero_count = gnz.size()
        # Reindex to ensure IDs with zero sales are included as NaN
        non_zero_count = non_zero_count.reindex(total_periods.index)
        adi = total_periods / non_zero_count

        # 4. Calculate CV² (Coefficient of Variation Squared)
        # Using vectorized mean and std is much faster than .apply(nz_stats)
        nz_mean = gnz.mean().reindex(total_periods.index)
        nz_std = gnz.std().reindex(total_periods.index)
        cv2 = (nz_std / nz_mean) ** 2

        # 5. Classification logic
        # Select requires arrays of the same length
        adi_arr = adi.values
        cv2_arr = cv2.values

        conditions = [
            (adi_arr < 1.32) & (cv2_arr < 0.49),
            (adi_arr < 1.32) & (cv2_arr >= 0.49),
            (adi_arr >= 1.32) & (cv2_arr < 0.49),
            (adi_arr >= 1.32) & (cv2_arr >= 0.49)
        ]
        
        adi_thresh = np.quantile(adi_arr, 0.6)
        cv2_thresh = np.quantile(cv2_arr, 0.6)
        conditions = [
            (adi_arr < adi_thresh) & (cv2_arr < cv2_thresh),
            (adi_arr < adi_thresh) & (cv2_arr >= cv2_thresh),
            (adi_arr >= adi_thresh) & (cv2_arr < cv2_thresh),
            (adi_arr >= adi_thresh) & (cv2_arr >= cv2_thresh)
        ]

        choices = ["Smooth", "Erratic", "Intermittent", "Lumpy"] # in that order
        
        segment = np.select(conditions, choices, default="Undefined")

        return pd.DataFrame({
            "id": adi.index,
            "ADI": adi_arr,
            "CV2": cv2_arr,
            "smoothness_segs": segment
        }).reset_index(drop=True)


    def prepare(self, path, cutoff_day, level=12):
        print(f"--- Loading: {path} ---")
        aggr_cols = self.level_map[level]
        schema_names    = set(pq.read_schema(path).names)
        target_on_disk  = "sold" if "sold" in schema_names else self.target
        needed = ({self.id, self.date, target_on_disk}
                  | (set(aggr_cols)        & schema_names)
                  | (set(self.covariates)  & schema_names)
                  | (set(self.static_cols) & schema_names))
        df = pd.read_parquet(path, columns=list(needed))
        if target_on_disk != self.target:
            df.rename(columns={target_on_disk: self.target}, inplace=True)
        df[self.date] = pd.to_datetime(df[self.date], utc=False, cache=True)
        for col in [c for c in df.columns if "event" in c]:
            if hasattr(df[col], "cat"):
                if "none" not in df[col].cat.categories:
                    df[col] = df[col].cat.add_categories("none")
                df[col] = df[col].fillna("none")
            else:
                df[col] = df[col].fillna("none").astype("category")
        type_map = {"snap": np.int8, "wday": np.int8, "month": np.int8,
                    "year": np.int16, "wm_yr_wk": np.int16,
                    "sell_price": np.float32, self.target: np.float32}
        for col, dtype in type_map.items():
            for c in [c for c in df.columns if col in c]:
                df[c] = df[c].astype(dtype)
        for col in aggr_cols + self.static_cols:
            if col in df.columns and df[col].dtype == object:
                df[col] = df[col].astype("category")
        print(f"--- Aggregating to Level {level} ---")
        agg_map = {self.target: "sum"}
        for col in self.covariates:
            if col in df.columns:
                if "snap" in col:   agg_map[col] = "max"
                elif "event" in col: agg_map[col] = "first"
                else:               agg_map[col] = "mean"
        for col in self.static_cols:
            if col in df.columns and col not in aggr_cols:
                agg_map[col] = "first"

        df = df.sort_values([*aggr_cols, self.date])
        df_aggr = (df.groupby(aggr_cols + [self.date], observed=True, sort=False)
                     .agg(agg_map).reset_index())
        del df
        gc.collect()
        
        if len(aggr_cols) == 0:
            df_aggr[self.id] = "All"
        elif len(aggr_cols) == 1:
            df_aggr[self.id] = df_aggr[aggr_cols[0]].astype(str)
        else:
            def _get_str_vals(s):
                if hasattr(s, "cat"):
                    return s.cat.categories.astype(str).values[s.cat.codes.values]
                return s.astype(str).values
            parts = [_get_str_vals(df_aggr[c]) for c in aggr_cols]
            ids   = parts[0]
            for part in parts[1:]:
                ids = np.char.add(np.char.add(ids, "_"), part)
            df_aggr[self.id] = ids
        df_aggr[self.id] = (df_aggr[self.id]
                             .astype(str)
                             .str.replace("_evaluation", "", regex=False)
                             .astype("category"))
        cutoff = pd.Timestamp(cutoff_day)

        df_aggr['id'] = df_aggr['id'].astype(str)

        #df_aggr = df_aggr.sort_values([self.id, self.date]).reset_index(drop=True)
        mask    = df_aggr[self.date] <= cutoff
        hist_df   = df_aggr.loc[mask]

        # add segments
        smoothness_segs = self.compute_smoothness_segments(hist_df)
        print(f"Segments computed. Merging into hist_df...")
        hist_df = hist_df.merge(smoothness_segs[["id", "smoothness_segs"]], on="id", how="left")

        future_df = df_aggr.loc[~mask]

        if self.target in future_df.columns:
            future_df.drop(columns=[self.target], inplace=True)
        static_present = [c for c in self.static_cols if c in df_aggr.columns]
        static_df = (df_aggr[[self.id] + static_present]
                       .drop_duplicates().reset_index(drop=True))
        print(f"Done. {df_aggr[self.id].nunique():,} series | "
              f"Hist: {len(hist_df):,} rows | Future: {len(future_df):,} rows")
        del df_aggr, smoothness_segs
        gc.collect()

        return hist_df, future_df, static_df

    def get_prepared_data(self, path, cutoff_day, level=12, force_reprepare=False):
        paths    = self._get_cache_paths(cutoff_day, level)
        use_cache = os.path.exists(paths["hist"]) and not force_reprepare
        if use_cache:
            print(f"--- Cache Hit: Data found in {paths['folder']} ---")
            return (pd.read_parquet(paths["hist"]),
                    pd.read_parquet(paths["future"]),
                    pd.read_parquet(paths["static"]))
        reason = "Force Reprepare" if force_reprepare else "Cache Miss"
        print(f"--- {reason}: Preparing Level {level} for {cutoff_day} ---")
        hist_df, future_df, static_df = self.prepare(path, cutoff_day, level)
        os.makedirs(paths["folder"], exist_ok=True)
        hist_df.to_parquet(paths["hist"],     index=False)
        future_df.to_parquet(paths["future"], index=False)
        static_df.to_parquet(paths["static"], index=False)
        return hist_df, future_df, static_df

dataprep_config = {"tag": DATA_TAG,
                   "base_cols": ['id', 'date', 'sales_quantity'],
                   "extra_cols": []
                }
pipeline = M5DataPipeline(config=dataprep_config)
hist_df, future_df, static_df = pipeline.get_prepared_data(DATA_PATH, CUTOFF_DAY, level=12, force_reprepare=True)

hist_df_trimmed = trim_series_to_active(hist_df, id_col="id", target_col="sales_quantity")
print(f"\nhist_df     : {hist_df.shape}  (columns: {list(hist_df.columns)})")
print(f"future_df   : {future_df.shape}  (columns: {list(future_df.columns)})")
print(f"static_df   : {static_df.shape}")

del pipeline
gc.collect()

--- Force Reprepare: Preparing Level 12 for 2016-04-24 ---
--- Loading: /mnt/lab/datasets/M5/jointed_M5.parquet ---
--- Aggregating to Level 12 ---
Segments computed. Merging into hist_df...
Done. 30,490 series | Hist: 58,327,370 rows | Future: 853,720 rows

hist_df     : (58327370, 19)  (columns: ['id', 'date', 'sales_quantity', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'smoothness_segs'])
future_df   : (853720, 17)  (columns: ['id', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'])
static_df   : (30490, 6)


0

In [7]:
# Load actuals for the evaluation phase (d_1942 – d_1969)
def transform_m5_ground_truth(gt_wide_df, calendar_path=CALENDAR_PATH):
    if "id" not in gt_wide_df.columns:
        gt_wide_df["id"] = gt_wide_df["item_id"] + "_" + gt_wide_df["store_id"] + "_evaluation"
    day_cols = [c for c in gt_wide_df.columns if c.startswith("d_")]
    df_long  = gt_wide_df.melt(id_vars=["id"], value_vars=day_cols,
                                var_name="d", value_name="sales_quantity")
    cal      = pd.read_csv(calendar_path)
    cal["date"] = pd.to_datetime(cal["date"])
    df_long["date"] = df_long["d"].map(dict(zip(cal["d"], cal["date"])))
    df_long["id"]   = df_long["id"].str.replace("_evaluation", "", regex=False)
    return df_long[["id", "date", "sales_quantity"]]


# Read the actual sales for the evaluation period. The file format differs between the two phases, so we handle them separately.

if CUTOFF_DAY == "2016-05-22": # limit for test dates

    # this as the held out test set for the M5 competition
    actuals_eval  = pd.read_csv(ACTUALS_PATH)
    df_actual = transform_m5_ground_truth(actuals_eval)
    print(f"Actuals loaded: {df_actual.shape} | dates: "
        f"{df_actual['date'].min().date()} → {df_actual['date'].max().date()}")
    
    del actuals_eval
    gc.collect()
else:
    path = "/mnt/lab/datasets/M5/jointed_M5.parquet"
    # read true sales for evaluation (can be used later in the notebook)
    df_actual = pd.read_parquet(path, columns=['id', 'date', 'sold']).rename(columns={'sold': 'sales_quantity'})
    df_actual["id"] = (
        df_actual["id"]
        .astype(str)
        .str.replace("_evaluation", "", regex=False)
        .str.replace("_validation", "", regex=False)
    )
    df_actual.head(3)



# 4 · Evaluator setup
Initialize the objects

In [8]:
def build_scale_smoothness_clusters(df_weights: pd.DataFrame,
                                    segments_df: pd.DataFrame) -> dict:
    # Merge once (avoid repeated joins later)
    df = df_weights.merge(
        segments_df[['id', 'smoothness_segs']].drop_duplicates(),
        on='id',
        how='inner'
    )

    # Define grouping logic
    dense = ["Smooth", "Erratic"]
    sparse = ["Intermittent", "Lumpy"]

    # Boolean masks (fast, vectorized)
    is_high = df["seller_type"] == "High"
    is_low  = df["seller_type"] == "Low"
    is_dense = df["smoothness_segs"].isin(dense)
    is_sparse = df["smoothness_segs"].isin(sparse)

    # Build clusters
    clusters = {
        "high_dense": df.loc[is_high & is_dense, "id"].unique(),
        "high_sparse": df.loc[is_high & is_sparse, "id"].unique(),
        "low_dense": df.loc[is_low & is_dense, "id"].unique(),
        "low_sparse": df.loc[is_low & is_sparse, "id"].unique(),
    }

    return clusters


In [9]:
evaluator = M5Evaluator(
    raw_train_df     = hist_df,         # full 1913-day untrimmed — for L1-L11 scales
    trimmed_train_df = hist_df_trimmed, # per-series trimmed — for L12 scales + weights
    static_df        = static_df,
    target_col       = "sales_quantity",
    price_col        = "sell_price",
)
suite = M5ExplorationSuite(
    horizon  = 28,
    ag_path  = "/mnt/lab/nmwamsojo/autogluon_models/explorations",
    base_dir = "/mnt/lab/nmwamsojo/prepared_data",
)

# prepare data
dataprep_config = {"tag": DATA_TAG}
pipeline = M5DataPipeline(config=dataprep_config)

paths    = pipeline._get_cache_paths(CUTOFF_DAY, level=12)
pipeline = M5DataPipeline(config={"tag": DATA_TAG})
hist_df, future_df, static_df = pipeline.get_prepared_data(
    DATA_PATH, CUTOFF_DAY, level=12 #, force_reprepare=True
)

level = 12
info = evaluator.level_info[level]
df_weights = pd.DataFrame({
    'scale': info['scales'],
    'weight': info['weights']
}, index=info['index']).reset_index()

hist_df_trimmed =trim_series_to_active(hist_df, id_col="id", target_col="sales_quantity")


# Adding binning for analysis
df_weights['seller_type'] = pd.qcut(df_weights['weight'], q=[0, 0.5, 1.0], labels=['Low', 'High'])

clusters = build_scale_smoothness_clusters(df_weights, hist_df_trimmed)
for k, v in clusters.items():
    print(f"{k}: {len(v)} series")

del hist_df, future_df, static_df, hist_df_trimmed
gc.collect()

  [CPU] CuPy not available — scale computation on 18 CPU cores.
  Building hierarchy scales and weights …
    Pivot [trimmed]: 30,490 series × 1,913 days
    Pivot [raw]: 30,490 series × 1,913 days
    Level  1 ready — 1 series
    Level  4 ready — 3 series
    Level  3 ready — 10 series
    Level  2 ready — 3 series
    Level  5 ready — 7 series
    Level  7 ready — 21 series
    Level  8 ready — 30 series
    Level  9 ready — 70 series
    Level  6 ready — 9 series
    Level 10 ready — 3,049 series
    Level 11 ready — 9,147 series
    Level 12 ready — 30,490 series
--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/default/level_12/20160424 ---
high_dense: 12110 series
high_sparse: 3135 series
low_dense: 6182 series
low_sparse: 9063 series


0

In [10]:
clusters.keys()

dict_keys(['high_dense', 'high_sparse', 'low_dense', 'low_sparse'])

# 5 · Run a single experiment
Change `EXP_TAG` and `EXP_CONFIG` to any entry from Section 2.

In [ ]:
import gc

import optuna
from optuna.samplers import TPESampler

# --- Chronos fine-tuned (LoRA, no covariates) ---------------------------------

covariate_options = {
    "event_1": ["event_name_1", "event_type_1"],
    "price": ["sell_price"],
    "only" : []
}

smoothness_seg = "high_dense" # we can also try "Erratic", "Intermittent", "Lumpy" or combinations like ["Smooth", "Erratic"]

_data_cache = {}


def objective(trial):
    # 1. Define the search space
    # Replace these with your desired ranges
    cov_type = trial.suggest_categorical("cov_type", ["only", "event_1", "price"])
    DATA_TAG = f"sales_{cov_type}"

    hp_dict = {
        "batch_size": trial.suggest_categorical("batch_size", [32, 256, 1024]),
        "context_length": trial.suggest_categorical("context_length", [2**i for i in range(0,10,1)]), # [32, 64, 128, 256, 512, 1024]    
        "known_cov_cols": covariate_options[cov_type],        # zeroshot ignores covariates
        # Other hyperparameters here:
        "fine_tune_steps": trial.suggest_int("fine_tune_steps", 500, 5000, step=500),
        "fine_tune_mode":  trial.suggest_categorical("fine_tune_mode", ["lora", "full"]),
        "fine_tune_lr":    trial.suggest_categorical("fine_tune_lr", [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]),
        "fine_tune_batch_size": trial.suggest_categorical("fine_tune_batch_size", [16, 32, 64, 128, 256, 512]),
    }

    # 2. Setup Experiment Tags and Config
    trial_tag = f"optuna_chronos2_trial_{trial.number}"

    # ── Choose ONE experiment to run ──────────────────────────────────────────────
    EXP_TAG = (
        f"hpo_chronos2_finetune"                       # e.g., _event_1
        f"_bs{hp_dict['batch_size']}"                       # e.g., 256
        f"_cl{hp_dict['context_length']}"       # e.g., _cl256
        f"_{hp_dict['fine_tune_mode']}"         # e.g., _lora
        f"_s{hp_dict['fine_tune_steps']}"       # e.g., _s1500
        f"_lr{hp_dict['fine_tune_lr']}"     # e.g., _lr1.45e-04
        f"_tfbs{hp_dict['fine_tune_batch_size']}"  # e.g., _bs256
    )
    
    # Create a local copy of the config to avoid bleeding state between trials
    current_config = CFG_CHRONOS_FINETUNE.copy()
    current_config.update(hp_dict)

    print(f"\n[Trial {trial.number}] Running experiment: {trial_tag}")
    print(f"\n[Trial {trial.number}] Running experiment: {EXP_TAG}")
    print(f"Cov type chosen: {cov_type} with covariates {covariate_options[cov_type]}")

    # 3. Run the Suite
    try:

        # prepare data
        dataprep_config = {"tag": DATA_TAG}
        pipeline = M5DataPipeline(config=dataprep_config)

        paths    = pipeline._get_cache_paths(CUTOFF_DAY, level=12)
        if not os.path.exists(paths["hist"]) or len(_data_cache) == 0:
            print("  [Data] Cache empty — running get_prepared_data()")
            pipeline = M5DataPipeline(config={"tag": DATA_TAG})
            hist_df, future_df, static_df = pipeline.get_prepared_data(
                DATA_PATH, CUTOFF_DAY, level=12
            )
            _data_cache["hist"]   = trim_series_to_active(hist_df, id_col="id", target_col="sales_quantity")
            _data_cache["future"] = future_df
            _data_cache["static"] = static_df
            del hist_df
        else:
            print("  [Data] Cache hit — skipping get_prepared_data()")

        hist_df_trimmed = _data_cache["hist"]
        future_df       = _data_cache["future"]
        static_df       = _data_cache["static"]

        del pipeline
        gc.collect()
        
        # fit & forecast
        fcst_df = suite.run(
            hist_df      = hist_df_trimmed[hist_df_trimmed.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            future_df    = future_df[future_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            static_df    = static_df[static_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            model        = "Chronos2",
            exp_config   = current_config,
            exp_tag      = EXP_TAG,
            data_tag     = DATA_TAG,
            cutoff_day   = CUTOFF_DAY,
            wrapper_dict = WRAPPER,
            force_run    = False
        )
        
        # evaluate
        metrics = evaluator.evaluate_all(fcst_df, df_actual)

        # clean up memory after each trial
        del fcst_df # free memory
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        return metrics['WRMSSE']

    except Exception as e:
        print(f"Trial failed due to: {e}")
        import traceback; traceback.print_exc()
        return float("inf") # Return infinity so Optuna ignores this failure

# 5. Initialize and Run the Study
study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=108, n_jobs=1, gc_after_trial=True) 

# 6. Results
print("\n\nBest hyperparameters:", study.best_params)

STUDY_NAME = f"chronos2_zeroshot_{smoothness_seg}"
save_dir = "/mnt/lab/nmwamsojo/prepared_data/hpo_studies/"
os.makedirs(save_dir, exist_ok=True)

df_trials = study.trials_dataframe()
csv_path = os.path.join(save_dir, f"{STUDY_NAME}_trials.csv")
df_trials.to_csv(csv_path, index=False)

del study
gc.collect()

[I 2026-05-06 07:33:42,858] A new study created in memory with name: no-name-7a0f65db-bdde-4aa1-a85d-4daa3e8dcf8b



[Trial 0] Running experiment: optuna_chronos2_trial_0

[Trial 0] Running experiment: hpo_chronos2_zeroshot_high_dense_bs1024_cl64
Cov type chosen: only with covariates []
  [Data] Cache empty — running get_prepared_data()
--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424 ---
[hpo_chronos2_zeroshot_high_dense_bs1024_cl64] Cache hit — loading forecasts from /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_chronos2_zeroshot_high_dense_bs1024_cl64/forecasts.parquet


[I 2026-05-06 07:34:27,956] Trial 0 finished with value: 2.180515 and parameters: {'cov_type': 'only', 'batch_size': 1024, 'context_length': 64}. Best is trial 0 with value: 2.180515.



[Trial 1] Running experiment: optuna_chronos2_trial_1

[Trial 1] Running experiment: hpo_chronos2_zeroshot_high_dense_bs1024_cl16
Cov type chosen: event_1 with covariates ['event_name_1', 'event_type_1']
  [Data] Cache hit — skipping get_prepared_data()
[hpo_chronos2_zeroshot_high_dense_bs1024_cl16] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_event_1/level_12/20160424/models/hpo_chronos2_zeroshot_high_dense_bs1024_cl16/forecasts.parquet…
[hpo_chronos2_zeroshot_high_dense_bs1024_cl16] Running Chronos2 experiment…
  known covariates : ['event_name_1', 'event_type_1']


fit known_cov_cols : ['event_name_1', 'event_type_1']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 16, 'device': 'cuda', 'batch_size': 1024}
[hpo_chronos2_zeroshot_high_dense_bs1024_cl16] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_event_1/level_12/20160424/models/hpo_chronos2_zeroshot_high_dense_bs1024_cl16/forecasts.parquet


...

[I 2026-05-06 07:55:15,712] Trial 33 finished with value: 1.438267 and parameters: {'cov_type': 'price', 'batch_size': 1, 'context_length': 1}. Best is trial 2 with value: 1.438267.



[Trial 34] Running experiment: optuna_chronos2_trial_34

[Trial 34] Running experiment: hpo_chronos2_zeroshot_high_dense_bs1_cl16
Cov type chosen: price with covariates ['sell_price']
  [Data] Cache hit — skipping get_prepared_data()


[hpo_chronos2_zeroshot_high_dense_bs1_cl16] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_price/level_12/20160424/models/hpo_chronos2_zeroshot_high_dense_bs1_cl16/forecasts.parquet…
[hpo_chronos2_zeroshot_high_dense_bs1_cl16] Running Chronos2 experiment…
  known covariates : ['sell_price']
fit known_cov_cols : ['sell_price']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 16, 'device': 'cuda', 'batch_size': 1}


[W 2026-05-06 07:55:23,728] Trial 34 failed with parameters: {'cov_type': 'price', 'batch_size': 1, 'context_length': 16} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/nmwamsojo/tsfm-explo/.venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/mnt/lab/nmwamsojo/uv_workspace/tmp/ipykernel_2670284/2475189692.py", line 87, in objective
    fcst_df = suite.run(
  File "/home/nmwamsojo/tsfm-explo/notebooks/m5_exploration.py", line 288, in run
    f_df = self._fit_predict(
  File "/home/nmwamsojo/tsfm-explo/notebooks/m5_exploration.py", line 627, in _fit_predict
    predictor = TimeSeriesPredictor(
  File "/home/nmwamsojo/tsfm-explo/.venv/lib/python3.10/site-packages/autogluon/common/utils/decorators.py", line 34, in _call
    return f(*gargs, **gkwargs)
  File "/home/nmwamsojo/tsfm-explo/.venv/lib/python3.10/site-packages/autogluon/timeseries/predictor.py", line

KeyboardInterrupt: 

In [ ]:
import gc

import optuna
from optuna.samplers import TPESampler

# --- Chronos fine-tuned (LoRA, no covariates) ---------------------------------

covariate_options = {
    "event_1": ["event_name_1", "event_type_1"],
    "price": ["sell_price"],
    "only" : []
}

smoothness_seg = "high_sparse" # we can also try "Erratic", "Intermittent", "Lumpy" or combinations like ["Smooth", "Erratic"]

_data_cache = {}


def objective(trial):
    # 1. Define the search space
    # Replace these with your desired ranges
    cov_type = trial.suggest_categorical("cov_type", ["only", "event_1", "price"])
    DATA_TAG = f"sales_{cov_type}"

    hp_dict = {
        "batch_size": trial.suggest_categorical("batch_size", [2**i for i in range(0,11,2)]),
        "context_length": trial.suggest_categorical("context_length", [2**i for i in range(0,11,2)]), # [32, 64, 128, 256, 512, 1024]    
        "known_cov_cols": covariate_options[cov_type],        # zeroshot ignores covariates
        # Other hyperparameters here:
        "fine_tune_steps": 0, #trial.suggest_int("fine_tune_steps", 500, 5000, step=500),
        "fine_tune_mode":  "lora", #trial.suggest_categorical("fine_tune_mode", ["lora", "full"]),
        #"fine_tune_lr":    trial.suggest_categorical("fine_tune_lr", [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]),
        #"fine_tune_batch_size": trial.suggest_categorical("fine_tune_batch_size", [64, 128, 256, 512]),
    }

    # 2. Setup Experiment Tags and Config
    trial_tag = f"optuna_chronos2_trial_{trial.number}"

    # ── Choose ONE experiment to run ──────────────────────────────────────────────
    # ── Choose ONE experiment to run ──────────────────────────────────────────────
    EXP_TAG = (
        f"hpo_chronos2_zeroshot_{smoothness_seg}"                       # e.g., _event_1
        f"_bs{hp_dict['batch_size']}"                       # e.g., 256
        f"_cl{hp_dict['context_length']}"       # e.g., _cl256
        #f"_{hp_dict['fine_tune_mode']}"         # e.g., _lora
        #f"_s{hp_dict['fine_tune_steps']}"       # e.g., _s1500
        #f"_lr{hp_dict['fine_tune_lr']}"     # e.g., _lr1.45e-04
        #f"_tfbs{hp_dict['fine_tune_batch_size']}"  # e.g., _bs256
    )
    
    # Create a local copy of the config to avoid bleeding state between trials
    current_config = CFG_CHRONOS_FINETUNE.copy()
    current_config.update(hp_dict)

    print(f"\n[Trial {trial.number}] Running experiment: {trial_tag}")
    print(f"\n[Trial {trial.number}] Running experiment: {EXP_TAG}")
    print(f"Cov type chosen: {cov_type} with covariates {covariate_options[cov_type]}")

    # 3. Run the Suite
    try:

        # prepare data
        dataprep_config = {"tag": DATA_TAG}
        pipeline = M5DataPipeline(config=dataprep_config)

        paths    = pipeline._get_cache_paths(CUTOFF_DAY, level=12)
        if not os.path.exists(paths["hist"]) or len(_data_cache) == 0:
            print("  [Data] Cache empty — running get_prepared_data()")
            pipeline = M5DataPipeline(config={"tag": DATA_TAG})
            hist_df, future_df, static_df = pipeline.get_prepared_data(
                DATA_PATH, CUTOFF_DAY, level=12
            )
            _data_cache["hist"]   = trim_series_to_active(hist_df, id_col="id", target_col="sales_quantity")
            _data_cache["future"] = future_df
            _data_cache["static"] = static_df
            del hist_df

        else:
            print("  [Data] Cache hit — skipping get_prepared_data()")

        hist_df_trimmed = _data_cache["hist"]
        future_df       = _data_cache["future"]
        static_df       = _data_cache["static"]

        del pipeline
        gc.collect()
        
        # fit & forecast
        fcst_df = suite.run(
            hist_df      = hist_df_trimmed[hist_df_trimmed.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            future_df    = future_df[future_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            static_df    = static_df[static_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            model        = "Chronos2",
            exp_config   = current_config,
            exp_tag      = EXP_TAG,
            data_tag     = DATA_TAG,
            cutoff_day   = CUTOFF_DAY,
            wrapper_dict = WRAPPER,
            force_run    = False
        )
        
        # evaluate
        metrics = evaluator.evaluate_all(fcst_df, df_actual)

        # clean up memory after each trial
        del fcst_df # free memory
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        return metrics['WRMSSE']

    except Exception as e:
        print(f"Trial failed due to: {e}")
        import traceback; traceback.print_exc()
        return float("inf") # Return infinity so Optuna ignores this failure

# 5. Initialize and Run the Study
study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=108, n_jobs=1, gc_after_trial=True) 

# 6. Results
print("\n\nBest hyperparameters:", study.best_params)

STUDY_NAME = f"chronos2_zeroshot_{smoothness_seg}"
save_dir = "/mnt/lab/nmwamsojo/prepared_data/hpo_studies/"
os.makedirs(save_dir, exist_ok=True)

df_trials = study.trials_dataframe()
csv_path = os.path.join(save_dir, f"{STUDY_NAME}_trials.csv")
df_trials.to_csv(csv_path, index=False)

del study
gc.collect()

In [ ]:
import gc

import optuna
from optuna.samplers import TPESampler

# --- Chronos fine-tuned (LoRA, no covariates) ---------------------------------

covariate_options = {
    "event_1": ["event_name_1", "event_type_1"],
    "price": ["sell_price"],
    "only" : []
}

smoothness_seg = "low_sparse" # we can also try "Erratic", "Intermittent", "Lumpy" or combinations like ["Smooth", "Erratic"]

_data_cache = {}


def objective(trial):
    # 1. Define the search space
    # Replace these with your desired ranges
    cov_type = trial.suggest_categorical("cov_type", ["only", "event_1", "price"])
    DATA_TAG = f"sales_{cov_type}"

    hp_dict = {
        "batch_size": trial.suggest_categorical("batch_size", [2**i for i in range(0,11,2)]),
        "context_length": trial.suggest_categorical("context_length", [2**i for i in range(0,11,2)]), # [32, 64, 128, 256, 512, 1024]    
        "known_cov_cols": covariate_options[cov_type],        # zeroshot ignores covariates
        # Other hyperparameters here:
        "fine_tune_steps": 0, #trial.suggest_int("fine_tune_steps", 500, 5000, step=500),
        "fine_tune_mode":  "lora", #trial.suggest_categorical("fine_tune_mode", ["lora", "full"]),
        #"fine_tune_lr":    trial.suggest_categorical("fine_tune_lr", [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]),
        #"fine_tune_batch_size": trial.suggest_categorical("fine_tune_batch_size", [64, 128, 256, 512]),
    }

    # 2. Setup Experiment Tags and Config
    trial_tag = f"optuna_chronos2_trial_{trial.number}"

    # ── Choose ONE experiment to run ──────────────────────────────────────────────
    # ── Choose ONE experiment to run ──────────────────────────────────────────────
    EXP_TAG = (
        f"hpo_chronos2_zeroshot_{smoothness_seg}"                       # e.g., _event_1
        f"_bs{hp_dict['batch_size']}"                       # e.g., 256
        f"_cl{hp_dict['context_length']}"       # e.g., _cl256
        #f"_{hp_dict['fine_tune_mode']}"         # e.g., _lora
        #f"_s{hp_dict['fine_tune_steps']}"       # e.g., _s1500
        #f"_lr{hp_dict['fine_tune_lr']}"     # e.g., _lr1.45e-04
        #f"_tfbs{hp_dict['fine_tune_batch_size']}"  # e.g., _bs256
    )
    
    # Create a local copy of the config to avoid bleeding state between trials
    current_config = CFG_CHRONOS_FINETUNE.copy()
    current_config.update(hp_dict)

    print(f"\n[Trial {trial.number}] Running experiment: {trial_tag}")
    print(f"\n[Trial {trial.number}] Running experiment: {EXP_TAG}")
    print(f"Cov type chosen: {cov_type} with covariates {covariate_options[cov_type]}")

    # 3. Run the Suite
    try:

        # prepare data
        dataprep_config = {"tag": DATA_TAG}
        pipeline = M5DataPipeline(config=dataprep_config)

        paths    = pipeline._get_cache_paths(CUTOFF_DAY, level=12)
        if not os.path.exists(paths["hist"]) or len(_data_cache) == 0:
            print("  [Data] Cache empty — running get_prepared_data()")
            pipeline = M5DataPipeline(config={"tag": DATA_TAG})
            hist_df, future_df, static_df = pipeline.get_prepared_data(
                DATA_PATH, CUTOFF_DAY, level=12
            )
            _data_cache["hist"]   = trim_series_to_active(hist_df, id_col="id", target_col="sales_quantity")
            _data_cache["future"] = future_df
            _data_cache["static"] = static_df
            del hist_df

        else:
            print("  [Data] Cache hit — skipping get_prepared_data()")

        hist_df_trimmed = _data_cache["hist"]
        future_df       = _data_cache["future"]
        static_df       = _data_cache["static"]

        del pipeline
        gc.collect()
        
        # fit & forecast
        fcst_df = suite.run(
            hist_df      = hist_df_trimmed[hist_df_trimmed.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            future_df    = future_df[future_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            static_df    = static_df[static_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            model        = "Chronos2",
            exp_config   = current_config,
            exp_tag      = EXP_TAG,
            data_tag     = DATA_TAG,
            cutoff_day   = CUTOFF_DAY,
            wrapper_dict = WRAPPER,
            force_run    = False
        )
        
        # evaluate
        metrics = evaluator.evaluate_all(fcst_df, df_actual)

        # clean up memory after each trial
        del fcst_df # free memory
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        return metrics['WRMSSE']

    except Exception as e:
        print(f"Trial failed due to: {e}")
        import traceback; traceback.print_exc()
        return float("inf") # Return infinity so Optuna ignores this failure

# 5. Initialize and Run the Study
study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=108, n_jobs=1, gc_after_trial=True) 

# 6. Results
print("\n\nBest hyperparameters:", study.best_params)

STUDY_NAME = f"chronos2_zeroshot_{smoothness_seg}"
save_dir = "/mnt/lab/nmwamsojo/prepared_data/hpo_studies/"
os.makedirs(save_dir, exist_ok=True)

df_trials = study.trials_dataframe()
csv_path = os.path.join(save_dir, f"{STUDY_NAME}_trials.csv")
df_trials.to_csv(csv_path, index=False)

del study
gc.collect()

In [ ]:
import gc

import optuna
from optuna.samplers import TPESampler

# --- Chronos fine-tuned (LoRA, no covariates) ---------------------------------

covariate_options = {
    "event_1": ["event_name_1", "event_type_1"],
    "price": ["sell_price"],
    "only" : []
}

smoothness_seg = "low_dense" # we can also try "Erratic", "Intermittent", "Lumpy" or combinations like ["Smooth", "Erratic"]

_data_cache = {}


def objective(trial):
    # 1. Define the search space
    # Replace these with your desired ranges
    cov_type = trial.suggest_categorical("cov_type", ["only", "event_1", "price"])
    DATA_TAG = f"sales_{cov_type}"

    hp_dict = {
        "batch_size": trial.suggest_categorical("batch_size", [2**i for i in range(0,11,2)]),
        "context_length": trial.suggest_categorical("context_length", [2**i for i in range(0,11,2)]), # [32, 64, 128, 256, 512, 1024]    
        "known_cov_cols": covariate_options[cov_type],        # zeroshot ignores covariates
        # Other hyperparameters here:
        "fine_tune_steps": 0, #trial.suggest_int("fine_tune_steps", 500, 5000, step=500),
        "fine_tune_mode":  "lora", #trial.suggest_categorical("fine_tune_mode", ["lora", "full"]),
        #"fine_tune_lr":    trial.suggest_categorical("fine_tune_lr", [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]),
        #"fine_tune_batch_size": trial.suggest_categorical("fine_tune_batch_size", [64, 128, 256, 512]),
    }

    # 2. Setup Experiment Tags and Config
    trial_tag = f"optuna_chronos2_trial_{trial.number}"

    # ── Choose ONE experiment to run ──────────────────────────────────────────────
    # ── Choose ONE experiment to run ──────────────────────────────────────────────
    EXP_TAG = (
        f"hpo_chronos2_zeroshot_{smoothness_seg}"                       # e.g., _event_1
        f"_bs{hp_dict['batch_size']}"                       # e.g., 256
        f"_cl{hp_dict['context_length']}"       # e.g., _cl256
        #f"_{hp_dict['fine_tune_mode']}"         # e.g., _lora
        #f"_s{hp_dict['fine_tune_steps']}"       # e.g., _s1500
        #f"_lr{hp_dict['fine_tune_lr']}"     # e.g., _lr1.45e-04
        #f"_tfbs{hp_dict['fine_tune_batch_size']}"  # e.g., _bs256
    )
    
    # Create a local copy of the config to avoid bleeding state between trials
    current_config = CFG_CHRONOS_FINETUNE.copy()
    current_config.update(hp_dict)

    print(f"\n[Trial {trial.number}] Running experiment: {trial_tag}")
    print(f"\n[Trial {trial.number}] Running experiment: {EXP_TAG}")
    print(f"Cov type chosen: {cov_type} with covariates {covariate_options[cov_type]}")

    # 3. Run the Suite
    try:

        # prepare data
        dataprep_config = {"tag": DATA_TAG}
        pipeline = M5DataPipeline(config=dataprep_config)

        paths    = pipeline._get_cache_paths(CUTOFF_DAY, level=12)
        if not os.path.exists(paths["hist"]) or len(_data_cache) == 0:
            print("  [Data] Cache empty — running get_prepared_data()")
            hist_df, future_df, static_df = pipeline.get_prepared_data(
                DATA_PATH, CUTOFF_DAY, level=12
            )
            _data_cache["hist"]   = trim_series_to_active(hist_df, id_col="id", target_col="sales_quantity")
            _data_cache["future"] = future_df
            _data_cache["static"] = static_df
            del hist_df

        else:
            print("  [Data] Cache hit — skipping get_prepared_data()")

        hist_df_trimmed = _data_cache["hist"]
        future_df       = _data_cache["future"]
        static_df       = _data_cache["static"]

        del pipeline
        gc.collect()
        
        # fit & forecast
        fcst_df = suite.run(
            hist_df      = hist_df_trimmed[hist_df_trimmed.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            future_df    = future_df[future_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            static_df    = static_df[static_df.id.isin(clusters[smoothness_seg])], # for the given smoothness segment
            model        = "Chronos2",
            exp_config   = current_config,
            exp_tag      = EXP_TAG,
            data_tag     = DATA_TAG,
            cutoff_day   = CUTOFF_DAY,
            wrapper_dict = WRAPPER,
            force_run    = False
        )
        
        # evaluate
        metrics = evaluator.evaluate_all(fcst_df, df_actual)

        # clean up memory after each trial
        del fcst_df # free memory
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        return metrics['WRMSSE']

    except Exception as e:
        print(f"Trial failed due to: {e}")
        import traceback; traceback.print_exc()
        return float("inf") # Return infinity so Optuna ignores this failure

# 5. Initialize and Run the Study
study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=108, n_jobs=1, gc_after_trial=True) 

# 6. Results
print("\n\nBest hyperparameters:", study.best_params)

STUDY_NAME = f"chronos2_zeroshot_{smoothness_seg}"
save_dir = "/mnt/lab/nmwamsojo/prepared_data/hpo_studies/"
os.makedirs(save_dir, exist_ok=True)

df_trials = study.trials_dataframe()
csv_path = os.path.join(save_dir, f"{STUDY_NAME}_trials.csv")
df_trials.to_csv(csv_path, index=False)

del study
gc.collect()